# Fall Detection �” End-to-End Pipeline

Single-driver notebook that runs the full Issue 001 �’ 002 �’ 003 pipeline
on a fresh Colab runtime. Opens with Drive mounted and closes with
WebDataset shards on Drive, ready for Issue 005 (Pipeline A training).

Stages:
  1. **Environment setup** �” mount Drive, install approved deps,
     verify Colab torch is intact, write the env lock + run log.
  2. **Kaggle credentials** �” read `KAGGLE_USERNAME` / `KAGGLE_KEY`
     from Colab Secrets (never logged, never written to Drive).
  3. **Stage URFD** �” pull the dataset once into Drive and skip on
     subsequent runs (idempotent).
  4. **Build URFD manifest** �” schema-1.1 manifest, one row per clip.
  5. **Run perception** �” YOLO26 + ByteTrack on every URFD clip's
     frame folder; write per-clip tracking artefacts to Drive.
  6. **Generate crop shards** �” WebDataset-style `.tar` shards of
     fixed-length person crop clips, metadata sidecar per frame.
  7. **Verify** �” sanity-check the shards round-trip and write a
     final summary.

**Before running**, add two Colab Secrets via the key icon on the
left sidebar:

  - `KAGGLE_USERNAME` �” your Kaggle username
  - `KAGGLE_KEY`      �” your Kaggle API key

Hard rules enforced by the modules this notebook drives:

  - Detector must be `yolo26m`; anything else aborts loudly.
  - Tracker must be `bytetrack.yaml` with `persist=True`.
  - Pretrained weights only �” no training.
  - No re-detection in the cropping module �” Issue 002 boxes only.
  - Crop clips are 32 ��” 224 ��” 224 by default, margin 0.30.
  - URFD cam0 = downstream path; cam1 stays as a hard slice.
  - Frozen-vault datasets (omnifall, caucafall, mcfd, fallvision)
    must NEVER appear in debug/train/validate.

## 1. Mount Drive + select data mode

We use **LOCAL mode** by default: datasets land on the Colab
local disk (``/content/fall_local``) so the active pipeline avoids
Google Drive FUSE small-file reads. Final artefacts still land on
Drive so they survive a runtime reset.

Override the mode via the ``FALL_DETECTION_DATA_MODE`` env var
(``local`` or ``drive``) or by passing a different root.

Drive is mounted regardless of mode because the artefact root
always lives there. The notebook never reads dataset frames from
Drive in LOCAL mode �” that's the whole point.


In [ ]:
import os, pathlib, sys, time

# Mount Drive for artefact persistence (always needed).
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not on Colab �” Drive mount skipped.")

PROJECT_ROOT = pathlib.Path(os.environ.get("FALL_DETECTION_PROJECT_ROOT", "/content/fall_detection"))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from colab.data_mode import (
    DataMode,
    DataLayout,
    resolve_data_layout,
    describe_layout,
)

DATA_MODE = DataMode(os.environ.get("FALL_DETECTION_DATA_MODE", "local").lower())
layout = resolve_data_layout(mode=DATA_MODE)
layout.ensure()

print(f"Data mode        : {layout.mode.value}")
print(f"Dataset root     : {layout.dataset_root}  (active reads happen here)")
print(f"Artefact root    : {layout.artifact_root}  (writes land here)")
print(f"Datasets dir     : {layout.datasets}")
print(f"Artefacts dir    : {layout.artifacts}")
print(f"Logs dir         : {layout.logs}")
print(f"Layout summary   : {describe_layout(layout)}")
if DATA_MODE is DataMode.LOCAL:
    print("Active processing reads from LOCAL disk; only artefacts persist on Drive.")


## 1a. Pipeline A skip / reuse config

Decides whether the heavy upstream build (URFD staging,
perception, crop generation) runs at all when Issue 003
crop shards already live on Drive.

Pipeline A (Issue 005 Step 1) only needs the cached
shards, and rebuilding them is the slow step. When the
shards exist, `Run All` should fast-forward to the
shard verification + Pipeline A loader smoke check
instead of repeating expensive work.

Set `FORCE_RECOMPUTE=True` to force the full rebuild
anyway -- useful after Issue 002 / 003 edits.

Idempotency key: actual `shard-*.tar` files on Drive.
No flag file -- Drive artefacts are the only source of
truth.


In [ ]:
# Pipeline A skip / reuse config.
#
# Default: if Issue 003 crop shards already exist on
# Drive, reuse them and fast-forward to the Pipeline A
# loader smoke check. Set FORCE_RECOMPUTE=True to force
# the full rebuild even when shards exist.

FORCE_RECOMPUTE: bool = False

CROPS_ROOT = layout.artifacts / "crops"
existing_shards = sorted(CROPS_ROOT.glob("shard-*.tar"))
CROPS_ALREADY_BUILT: bool = bool(existing_shards) and not FORCE_RECOMPUTE

print(f"CROPS_ROOT                : {CROPS_ROOT}")
print(f"Existing shards on Drive : {len(existing_shards)}")
print(f"FORCE_RECOMPUTE           : {FORCE_RECOMPUTE}")
if CROPS_ALREADY_BUILT:
    print("Notebook will REUSE existing crop shards and fast-forward "
          "to shard verification + Pipeline A smoke check.")
else:
    if FORCE_RECOMPUTE:
        print("FORCE_RECOMPUTE=True: full rebuild will run even "
              "though shards exist on Drive.")
    else:
        print("No crop shards found: full build (staging + "
              "perception + cropping) will run automatically.")


## 2. Install approved dependencies

Installs the curated dependency stack (kagglehub, ultralytics,
transformers, …). **Does NOT** reinstall torch or torchvision �”
those come pre-built from Colab's CUDA-matched image.

In [ ]:
from colab.setup import run_setup
artifacts = run_setup(
    install_deps=True,
    write_freeze=True,
    write_run_log=True,
    extra_log_fields={"issue": "000_full_pipeline", "notebook": "colab/000_full_pipeline.ipynb"},
)
for name, path in artifacts.items():
    print(f"  {name:<12} -> {path}")

## 3. Verify Colab's torch is intact

Catches the silent-CPU-only break early.

In [ ]:
import torch
print(f"torch version     : {torch.__version__}")
print(f"torch CUDA version: {torch.version.cuda}")
print(f"CUDA available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device count      : {torch.cuda.device_count()}")
    print(f"Device 0          : {torch.cuda.get_device_name(0)}")
assert torch.cuda.is_available(), "CUDA not available �” check Colab runtime type."
assert torch.backends.cudnn.is_available(), "cuDNN not available �” torch is probably broken."

## 4. Stage URFD from the authoritative university source

URFD now comes from the University of Rzeszów page
(<https://fenix.ur.edu.pl/~mkepski/ds/data/>). No Kaggle
credentials are required. Frames stage on a local fast disk
(``layout.root``) and the two label CSVs persist on Drive
under ``layout.artifact_root / artifacts / urfd_labels`` so
they survive a fresh `Run All` without a re-download.


In [ ]:
if not CROPS_ALREADY_BUILT:
    from data.stage_urfd_university import stage_urfd_from_university

    # Frames go on the local disk; CSVs persist on Drive.
    # The university stager downloads every cam0 RGB zip
    # (30 fall zips + 40 adl zips) plus the two label CSVs.
    staging = stage_urfd_from_university(
        layout.root,
        csv_root=layout.artifact_root,
    )
    print(f"URFD staged at: {staging.staged_root}")
    print(f"  clip folders : {staging.clip_count}")
    print(f"  already staged: {staging.already_staged}")
    print(f"  source base URL: {staging.source_base_url}")
    print()
    print("CSV label paths (persistent on artefact root):")
    for name, path in sorted(staging.csv_paths.items()):
        print(f"  {name:<32}  {path}")
    print()
    print("Sample clip folders:")
    for folder in staging.clip_folders[:8]:
        print(f"  {folder.folder_name:<28}  label={folder.label:<8}  camera={folder.camera}  seq={folder.clip_sequence}")
else:
    print("Reusing existing crop shards on Drive; skipping Kaggle-free university staging. Set FORCE_RECOMPUTE=True (in the §1a config cell) to rebuild anyway.")


## 5. URFD now comes from the university source

Idempotent: short-circuits when the marker + every
expected file is present across BOTH the frame staging
root (frames + marker) AND the persistent CSV root
(labels). Re-runs without `FORCE_RECOMPUTE=True` never
re-download a multi-GB dataset. ``FORCE_RECOMPUTE=True`
clears the frame staging root + the persistent CSV root
before re-downloading so a half-staged tree is never
marked complete.

The university page publishes every fall/adl cam0 zip in
the form ``<seq>-cam0-rgb.zip``. Each zip extracts to a
double-nested ``<staged_root>/<seq>-cam0-rgb/<seq>-cam0-rgb/*.png``
layout (160 frames per clip, 1-based 3-digit zero-padded
filenames). The existing :class:`perception.frames.FrameFolderReader`
already descends into that nested shape.


In [ ]:
if not CROPS_ALREADY_BUILT:
    pass  # staging already happened in cell 10 - nothing to do here
else:
    print("Reusing existing crop shards on Drive; URFD staging skipped. Set FORCE_RECOMPUTE=True (in the §1a config cell) to rebuild anyway.")


## 6. Build the URFD debug manifest

Walks the staged tree, parses each folder name, emits a schema-1.1
manifest. Validates before write �” a malformed staged tree can never
produce a broken manifest.

In [ ]:
if not CROPS_ALREADY_BUILT:
    from data.build_urfd_manifest import build_urfd_manifest, write_urfd_manifest

    manifest = build_urfd_manifest(staging.staged_root)
    manifest_path = staging.staged_root / "manifest.yaml"
    write_urfd_manifest(manifest, manifest_path)
    fall_count = sum(1 for c in manifest.clips if c.label.value == "fall")
    no_fall_count = sum(1 for c in manifest.clips if c.label.value == "no_fall")
    print(f"Manifest: {manifest_path}")
    print(f"  schema version : {manifest.schema_version}")
    print(f"  clip rows      : {len(manifest.clips)}")
    print(f"  fall labels    : {fall_count}")
    print(f"  no_fall labels : {no_fall_count}")
else:
    print('Reusing existing crop shards on Drive; heavy upstream build is skipped. Set FORCE_RECOMPUTE=True (in the config cell above) to rebuild anyway.')



## 7. Run YOLO26 + ByteTrack on every URFD clip

**Important �” perception reads from local disk, not Drive.**

Live runs showed identical ~1.8 fps on L4 and A100 GPUs. The GPU
was not the bottleneck; Drive FUSE small-file reads were leaving
the GPU idle. We now copy each clip's frames from Drive to
`/content/fall_detection_local/<clip_id>/` BEFORE tracking, then
run the tracker against the local copy. Only the project's
artefact paths on Drive are written back; the raw dataset folder
on Drive is never modified.

**Compute policy:** Issue 002 should run on a **T4** with
local-disk frames. The T4 + local-disk combo is sufficient for
YOLO26 person detection and ByteTrack tracking at production-
relevant FPS. A100 is **reserved** for compute-bound work,
especially VideoMAE fine-tuning in Issue 006 if it ends up needed;
running Issue 002 on A100 does not help (we just observed ~1.8
fps on both).

For each clip we record:

  - Drive-to-local copy time and throughput (frames/sec).
  - Tracking runtime.
  - Total runtime.
  - FPS after local staging.
  - GPU name + CUDA version + runtime type.

Strict model + tracker checks; no fallback levers applied by
default (baseline is what runs unless proven weak on real footage).
Full-length tracks (`MAX_FRAMES_PER_CLIP=None`) so Issue 003 sees
untruncated clips.


In [ ]:
if not CROPS_ALREADY_BUILT:
    import time
    from pathlib import Path
    from perception.local_staging import (
        LocalFrameStager,
        COLAB_LOCAL_ROOT_DEFAULT,
        SKIP_STAGING_ENV_VAR,
    )
    from perception.frames import FrameFolderReader
    from perception.tracker import (
        TrackerConfig,
        run_tracker_on_folder,
        query_gpu_name,
    )
    from perception.report import build_track_continuity_report
    from perception.artifacts import (
        write_perception_artifacts,
        detections_grouped_by_frame,
    )
    from perception.render import render_annotated_clip
    from PIL import Image
    import numpy as np
    import torch

    MAX_FRAMES_PER_CLIP = None  # None = full-length (required by Issue 003); set a small int only for smoke-testing.
    ARTIFACTS_ROOT = layout.artifacts / "perception"
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

    GPU_NAME = query_gpu_name() or 'none (CPU-only runtime)'
    RUNTIME_TYPE = 'T4' if GPU_NAME and 'T4' in GPU_NAME else (
        'A100' if GPU_NAME and 'A100' in GPU_NAME else (
        'L4' if GPU_NAME and 'L4' in GPU_NAME else 'unknown GPU'))
    print(f"GPU detected     : {GPU_NAME}")
    print(f"Runtime type     : {RUNTIME_TYPE}")
    print(f"torch CUDA       : {torch.version.cuda}")
    print("Compute policy   : Issue 002 should run on T4; A100 reserved for compute-bound work (VideoMAE fine-tuning)")
    print(f"Local staging    : {COLAB_LOCAL_ROOT_DEFAULT}  (set {SKIP_STAGING_ENV_VAR}=1 to skip)")
    print()

    config = TrackerConfig()
    summary_rows = []
    stager = LocalFrameStager(local_root=COLAB_LOCAL_ROOT_DEFAULT)

    pipeline_started = time.perf_counter()

    for clip in manifest.clips:
        clip_folder = layout.root / clip.source_path
        if not clip_folder.is_dir():
            print(f"[skip] {clip.clip_id}: folder missing on Drive: {clip_folder}")
            continue

        # 1. Discover frames on Drive.
        reader = FrameFolderReader(clip_folder)
        ordered = reader.frames()
        if not ordered:
            print(f"[skip] {clip.clip_id}: no frames in {clip_folder}")
            continue

        capped = ordered if MAX_FRAMES_PER_CLIP is None else ordered[:MAX_FRAMES_PER_CLIP]

        # 2. Stage to local disk (the whole point of this fix).
        with stager.stage_clip(clip.clip_id, clip_folder, capped) as staged:
            local_paths = [Path(p) for p in sorted(staged.local_folder.iterdir())]
            local_frame_count = len(local_paths)
            copy_elapsed = staged.result.elapsed_seconds

            # 3. Track from local disk.
            print(f"[run]  {clip.clip_id}: staged {local_frame_count} frames "
                  f"in {copy_elapsed:.1f}s ({staged.result.fps_copy:.0f} frames/s); "
                  f"GPU={GPU_NAME}")
            track_started = time.perf_counter()
            run = run_tracker_on_folder(clip.clip_id, local_paths, config=config)
            track_elapsed = time.perf_counter() - track_started
            run.source_folder = str(clip_folder.relative_to(layout.root))

            report = build_track_continuity_report(run, source_folder=run.source_folder)
            out_dir = ARTIFACTS_ROOT / clip.clip_id
            write_perception_artifacts(out_dir, run, report)

            # 4. Render annotated frames (local) for visual review.
            images = [np.array(Image.open(p).convert("RGB")) for p in local_paths]
            detections_by_frame = detections_grouped_by_frame(run.detections, run.frame_count)
            render_annotated_clip(
                images, detections_by_frame,
                output_dir=out_dir / "annotated",
                clip_id=clip.clip_id,
            )

            total_clip = copy_elapsed + track_elapsed
            summary_rows.append({
                "clip_id": clip.clip_id,
                "frames": run.frame_count,
                "detections": run.detection_count,
                "tracks": run.track_count,
                "longest_track_id": report.longest_track_id,
                "longest_track_length": report.longest_track_length,
                "id_switch_count": report.id_switch_count,
                "decode_failures": run.decode_failures,
                "copy_seconds": round(copy_elapsed, 2),
                "copy_fps": round(staged.result.fps_copy, 1),
                "track_seconds": round(track_elapsed, 2),
                "track_fps": round(report.fps, 2),
                "total_clip_seconds": round(total_clip, 2),
            })
            print(f"       -> copy {copy_elapsed:.1f}s | track {track_elapsed:.1f}s | total {total_clip:.1f}s")
            print(f"       -> {report.detection_count} detections, "
                  f"{report.track_count} tracks, longest=track {report.longest_track_id} "
                  f"({report.longest_track_length} frames), "
                  f"{report.id_switch_count} id switches, {report.fps:.1f} fps")
            print(f"       -> artefacts on Drive: {out_dir}")
            # 5. Context manager exit removes the local copy automatically.

    pipeline_elapsed = time.perf_counter() - pipeline_started
    print()
    print(f"Pipeline finished in {pipeline_elapsed:.1f}s.")

else:
    print('Reusing existing crop shards on Drive; heavy upstream build is skipped. Set FORCE_RECOMPUTE=True (in the config cell above) to rebuild anyway.')



## 8. Perception summary

Two human-review questions:

  1. **Did the falling person keep the same track_id through the fall
     window?** Open the annotated frames under
     `artifacts/perception/<clip_id>/annotated/` and check.

  2. **Did ID switches happen at the moment of collapse?** Look at
     `id_switch_count` for fall clips. Non-zero on a fall clip is
     the known risk called out in the PRD *Further Notes*.

If a fall clip shows zero detections, or the longest track does NOT
span the fall, apply the fallback levers in escalation order:

  1. Lower `track_low_thresh` (more permissive association).
  2. Switch to BoT-SORT (`fallback_tracker='botsort.yaml'`).
  3. Try `fallback_end2end=False` (manual code change �” BoT-SORT only).

In [ ]:
if not CROPS_ALREADY_BUILT:
    import json

    summary_path = ARTIFACTS_ROOT / "_run_summary.json"
    summary_path.write_text(json.dumps({
        "gpu_name": GPU_NAME,
        "runtime_type": RUNTIME_TYPE,
        "torch_cuda": torch.version.cuda,
        "compute_policy": "Issue 002 should run on T4; A100 reserved for compute-bound work (VideoMAE fine-tuning in Issue 006)",
        "local_staging_root": str(COLAB_LOCAL_ROOT_DEFAULT),
        "max_frames_per_clip": MAX_FRAMES_PER_CLIP,
        "config": {
            "model_name": config.model_name,
            "tracker_config": config.tracker_config,
            "confidence_threshold": config.confidence_threshold,
        },
        "metric_availability": {
            "map_50": "n/a (no detection ground truth)",
            "idf1": "n/a (no tracking ground truth)",
            "mota": "n/a (no tracking ground truth)",
            "hota": "n/a (no tracking ground truth)",
        },
        "pipeline_elapsed_seconds": round(pipeline_elapsed, 2),
        "clips": summary_rows,
    }, indent=2), encoding="utf-8")
    print(f"Perception summary: {summary_path}")
    print()
    print(f"  GPU            : {GPU_NAME}")
    print(f"  Runtime type   : {RUNTIME_TYPE}")
    print(f"  torch CUDA     : {torch.version.cuda}")
    print(f"  Total pipeline : {pipeline_elapsed:.1f}s")
    print()
    print(f"  {'clip_id':<32} {'frames':>7} {'dets':>6} {'tracks':>7} {'longest':>8} {'switches':>9} {'copy_s':>7} {'track_s':>8} {'fps':>6} {'dec_fail':>9}")
    for row in summary_rows:
        print(f"  {row['clip_id']:<32} {row['frames']:>7} {row['detections']:>6} "
              f"{row['tracks']:>7} {str(row['longest_track_id']):>8} {row['id_switch_count']:>9} "
              f"{row['copy_seconds']:>7.1f} {row['track_seconds']:>8.1f} {row['track_fps']:>6.1f} {row['decode_failures']:>9}")

else:
    print('Reusing existing crop shards on Drive; heavy upstream build is skipped. Set FORCE_RECOMPUTE=True (in the config cell above) to rebuild anyway.')



## 9. Configure crop parameters

Defaults match the PRD's recommended fixed clip contract.

In [ ]:
if not CROPS_ALREADY_BUILT:
    from cropping.clip_builder import CropConfig

    crop_config = CropConfig(
        output_size=224,   # 224x224 (PRD default)
        margin=0.30,       # mid-range of PRD [0.20, 0.40]
        clip_length=32,    # 32 frames (PRD: 16 or 32)
    )
    print(f"Crop config: output_size={crop_config.output_size}, "
          f"margin={crop_config.margin}, clip_length={crop_config.clip_length}")
else:
    print('Reusing existing crop shards on Drive; heavy upstream build is skipped. Set FORCE_RECOMPUTE=True (in the config cell above) to rebuild anyway.')



## 10. Generate WebDataset-style crop shards

Filter to URFD cam0 tracks per the Issue 002 close-out decision
(cam0 = downstream path; cam1 stays as a hard slice). Consume
Issue 002 tracking boxes verbatim �” no re-detection.

Output goes to `artifacts/crops/` on Drive.

**Important �” cropping also reads from local disk, not Drive.**

The cropping step reads each clip's source PNG frames to build the
crop tensor for every frame in every window. Reading directly from
Drive re-introduces the same FUSE small-file I/O bottleneck the
Issue 002 perception fix removed. We therefore stage each clip's
frames to `<local_root>/crops_<clip_id>/` before reading, mirroring
the perception step. The local copy is removed automatically when
the clip finishes (defensive `try / finally` so a tracker exception
doesn't leave stale frames on local disk).

Set `FALL_DETECTION_SKIP_LOCAL_STAGING=1` to skip even when a
`local_root` is set �” useful for hosts that already have fast disk
access to the data.


In [ ]:
if not CROPS_ALREADY_BUILT:

    if FORCE_RECOMPUTE:
        # Clear stale 21-clip Kaggle-derived shards so the
        # new 70-clip university build does not leave a
        # half-staged tree behind.
        if CROPS_ROOT.exists():
            shutil.rmtree(CROPS_ROOT)
    from perception.local_staging import COLAB_LOCAL_ROOT_DEFAULT
    from cropping.runner import run_cropping, write_run_summary

    PERCEPTION_ROOT = layout.artifacts / "perception"
    CROPS_ROOT = layout.artifacts / "crops"
    CROPS_ROOT.mkdir(parents=True, exist_ok=True)

    crop_summary = run_cropping(
        layout_root=layout.root,
        perception_root=PERCEPTION_ROOT,
        crops_root=CROPS_ROOT,
        manifest_path=manifest_path,
        crop_config=crop_config,
        camera_filter="cam0",  # Issue 002 close-out decision
        max_shards=9999,
        local_root=COLAB_LOCAL_ROOT_DEFAULT,  # local-disk staging (Issue 002 fix)
    )

    crop_summary_path = CROPS_ROOT / "_run_summary.json"
    write_run_summary(crop_summary, crop_summary_path)
    print(f"Cropping summary: {crop_summary_path}")
    print(f"  clips processed     : {crop_summary.clips_processed}")
    print(f"  tracks processed    : {crop_summary.tracks_processed}")
    print(f"  windows emitted     : {crop_summary.windows_emitted}")
    print(f"  windows skipped     : {crop_summary.windows_skipped}")
    print(f"  skip reasons        : {crop_summary.skip_reason_counts}")
    print(f"  shards written      : {crop_summary.shards_written}")
    print(f"  local staging root  : {crop_summary.local_staging_root}")
    print(f"  elapsed seconds     : {crop_summary.elapsed_seconds:.1f}")

else:
    print('Reusing existing crop shards on Drive; heavy upstream build is skipped. Set FORCE_RECOMPUTE=True (in the config cell above) to rebuild anyway.')




## 11. Verify a shard round-trips

Reads one shard back, confirms image + JSON sidecar pairs and the
per-shard manifest are present and complete.

In [ ]:
from cropping.shard_writer import read_shard

shards = sorted(CROPS_ROOT.glob("shard-*.tar"))
if not shards:
    print("No shards written �” nothing to verify.")
else:
    sample = shards[0]
    result = read_shard(sample)
    print(f"Sampled: {sample.name}")
    print(f"  image members : {len(result.image_members)}")
    print(f"  meta members  : {len(result.metadata_members)}")
    print(f"  manifest keys : {sorted(result.manifest.keys())}")
    sample_meta_member = next(iter(result.metadata_members))
    sample_meta = result.metadata_members[sample_meta_member]
    print()
    print(f"Sample metadata ({sample_meta_member}):")
    for key in ("clip_id", "dataset", "label", "track_id",
                "frame_index", "coverage", "crop_config",
                "margin_used", "shard_filename"):
        print(f"  {key:<15}: {sample_meta.get(key)}")

## 11a. URFD CSV / crop-window labeling alignment

A real-shard smoke check that connects the cached crop
windows to the persistent URFD CSV labels via
:func:`data.urfd_labels.label_window`. The label CSV is
NOT baked into the shard - labels are looked up at training
time. The cell reads actual frame_index values from the
crop metadata (not inferred from `offset * 32 + 1`).

Expected pattern for `urfd-debug-fall-01-cam0-rgb`:
  - early window (frames 1..32) → ``no_fall``
  - mid-fall window (spanning frame 83+ onset) → ``fall``

Expected pattern for `urfd-debug-adl-01-cam0-rgb`:
  - all windows → ``no_fall``


In [ ]:
# Labeling-alignment smoke check: connect crop windows to the URFD CSV
# labels by looking up each window's ACTUAL frame_index values (read from
# the crop metadata) against the persistent per-frame label CSVs. This is
# a smoke check only -- it does NOT train or compute model metrics.
from data.urfd_labels import parse_urfd_csv_label_file, label_window
from data.stage_urfd_university import FALL_CSV_FILENAME, ADL_CSV_FILENAME
from cropping.shard_writer import read_shard

_csv_dir = layout.artifact_root / "artifacts" / "urfd_labels"
_fall_csv = _csv_dir / FALL_CSV_FILENAME
_adls_csv = _csv_dir / ADL_CSV_FILENAME

if not shards:
    print("No crop shards on disk; skipping labeling-alignment smoke check.")
elif not (_fall_csv.is_file() and _adls_csv.is_file()):
    print(f"URFD label CSVs not present at {_csv_dir}; skipping labeling-alignment smoke check.")
else:
    _fall_labels = parse_urfd_csv_label_file(_fall_csv)
    _adl_labels = parse_urfd_csv_label_file(_adls_csv)

    def _window_frames(result, window_key):
        # Collect (frame_offset, frame_index, clip_id) for this window's
        # frames, ordered by offset. frame_index is the ORIGINAL 1-based
        # source frame number that aligns with the CSV frame_number.
        rows = []
        for _name, _meta in result.metadata_members.items():
            if _name.startswith(window_key + "_") and "frame_index" in _meta:
                rows.append((int(_meta["frame_offset"]),
                             int(_meta["frame_index"]),
                             _meta.get("clip_id")))
        rows.sort()
        frames = [fi for _, fi, _ in rows]
        clip_id = rows[0][2] if rows else None
        return frames, clip_id

    print("URFD CSV / crop-window labeling alignment (sample fall + adl windows):")
    _n_fall, _n_adl = 0, 0
    for _shard_path in shards:
        _result = read_shard(_shard_path)
        for _window_key in _result.manifest.get("clip_keys", []):
            _frames, _clip_id = _window_frames(_result, _window_key)
            if not _frames or not _clip_id:
                continue
            _is_fall = "fall" in _clip_id
            if _is_fall and _n_fall >= 3:
                continue
            if (not _is_fall) and _n_adl >= 2:
                continue
            _csv = _fall_labels if _is_fall else _adl_labels
            _label, _conf = label_window(_clip_id, _frames, _csv)
            print(f"  {_window_key:<44} frames {min(_frames):>3}-{max(_frames):>3}"
                  f"  -> {_label:<9} confuser={_conf}")
            if _is_fall:
                _n_fall += 1
            else:
                _n_adl += 1
        if _n_fall >= 3 and _n_adl >= 2:
            break
    print("Expected: fall windows before frame 83 -> no_fall; "
          "windows spanning 83+ -> fall; adl windows -> no_fall.")


## 12. Pipeline A loader smoke check

Confirms the Issue 005 Step 1 data-prep loader reads real
Issue 003 crop shards into VideoMAE-ready ``(T, 3, H, W)``
float32 tensors with the pinned ImageNet normalisation.

This is a **smoke check**, not a training step. The notebook
does NOT load VideoMAE here �” only the loader. Pipeline A
training lands in a later Issue 005 step.


In [ ]:
from pipeline_a import (
    list_clip_keys,
    load_clip_batch_from_shards,
    read_videomae_constants,
    ALLOWED_T,
    IMAGE_SIZE,
    NUM_CHANNELS,
    IMAGENET_MEAN,
    IMAGENET_STD,
)

if not shards:
    raise SystemExit(
        "No crop shards on disk yet �” Issue 003 must run before this smoke check."
    )

# Read clip_length from the actual sample shard's metadata
# rather than relying on the loader's DEFAULT_T. Real Issue 003
# shards are written at crop_length=32; if a future shard is
# written at 16, the smoke check must follow.
_sample_meta = next(iter(read_shard(shards[0]).metadata_members.values()))
_T = int(_sample_meta["crop_config"]["clip_length"])
assert _T in ALLOWED_T, (
    f"Shard clip_length {_T} not in Pipeline A allowed T={ALLOWED_T}; "
    f"resize Issue 003 clip_length to {ALLOWED_T} or extend the loader."
)

cfg = read_videomae_constants()
print("Pipeline A constants pinned to:")
for key, value in cfg.items():
    print(f"  {key:<14} = {value}")
print(f"Inferred clip_length (T) from shard metadata: {_T}")

all_window_keys = []
for shard in shards:
    all_window_keys.extend(list_clip_keys(shard))
    if len(all_window_keys) >= 6:
        break
sample_keys = all_window_keys[:4]
print(f"\nCollected {len(all_window_keys)} window keys across {len(shards)} shard(s); sampling {len(sample_keys)} for the smoke check.")

if not sample_keys:
    raise SystemExit("No windows in the available shards.")

batch, clips = load_clip_batch_from_shards(
    shards, clip_keys=sample_keys, T=_T,
)

print()
print("Tensor contract check:")
print(f"  batch shape   : {batch.shape}")
print(f"  expected      : (B={len(sample_keys)}, T={_T}, C={NUM_CHANNELS}, H={IMAGE_SIZE}, W={IMAGE_SIZE})")
assert batch.shape == (len(sample_keys), _T, NUM_CHANNELS, IMAGE_SIZE, IMAGE_SIZE), (
    f"Pipeline A tensor contract violated: {batch.shape}"
)
print(f"  batch dtype   : {batch.dtype}  (expected float32)")
assert batch.dtype == "float32"

print(f"  batch min/max : {batch.min():.3f} / {batch.max():.3f}")
assert batch.min() < 0.0 and batch.max() > 1.0, (
    "Normalisation contract violated: tensors look like raw [0, 1]."
)

print("\nPer-window identity:")
for c in clips:
    print(f"  clip_key={c.clip_key:<28}  bare clip_id={c.clip_id}  label={c.label}  track_id={c.track_id}")

first = clips[0].frames[0]
print(f"\nVisual sanity (clips[0].frames[0]):")
print(f"  dtype          : {first.dtype}")
print(f"  shape          : {first.shape}")
print(f"  channel means  : R={first[..., 0].mean():.1f} G={first[..., 1].mean():.1f} B={first[..., 2].mean():.1f}")
print(f"  label          : {clips[0].label}")

_processor_mean = None
_processor_std = None
try:
    from transformers import VideoMAEImageProcessor
    _proc = VideoMAEImageProcessor.from_pretrained(cfg["model_id"])
    _processor_mean = tuple(_proc.image_mean)
    _processor_std = tuple(_proc.image_std)
except Exception as _exc:
    print(f"\nVideoMAE parity check skipped: {_exc}")

if _processor_mean is not None:
    assert tuple(IMAGENET_MEAN) == _processor_mean, (
        f"IMAGENET_MEAN {IMAGENET_MEAN} != processor {_processor_mean}"
    )
    assert tuple(IMAGENET_STD) == _processor_std, (
        f"IMAGENET_STD {IMAGENET_STD} != processor {_processor_std}"
    )
    print(f"\nVideoMAE parity: IMAGENET_MEAN / IMAGENET_STD match {cfg['model_id']}.")

print()
print(f"Pipeline A loader smoke check passed at T={_T}.")


## 13. Done �” Issue 002 + 003 closure conditions

All Issue 001 �’ 003 deliverables are on Drive:

  - `MyDrive/fall_detection/datasets/urfd/` �” staged URFD frames.
  - `MyDrive/fall_detection/datasets/urfd/manifest.yaml` �” URFD debug manifest (schema 1.1).
  - `MyDrive/fall_detection/artifacts/perception/<clip_id>/` �” per-clip tracking outputs + annotated frames.
  - `MyDrive/fall_detection/artifacts/crops/shard-NNNNNN.tar` �” WebDataset-style crop shards.
  - `MyDrive/fall_detection/artifacts/crops/_run_summary.json` �” cropping summary.
  - `MyDrive/fall_detection/artifacts/pip_freeze.txt` �” environment lock.
  - `MyDrive/fall_detection/logs/setup_run_log.json` �” setup run log.

**Issue 002 closure condition** (per the review): visually inspect
annotated frames on Drive. The falling person must keep the same
`track_id` through the fall window.

**Issue 003 closure condition** (per the review): inspect a few
crops from each role in the shards. The person should be visible
and centred, not chopped or shrunk to zero �” catches detection /
tracking output quality on real footage that unit tests cannot.

**Pipeline A smoke check (Issue 005 Step 1):** the cell above
reads a small batch of windows from the cached shards into
VideoMAE-ready ``(T, 3, H, W)`` float32 tensors. A passing
smoke check confirms the on-disk shard format is what the
loader expects and the normalisation is the pinned-videomae-
base contract.

Once both visual reviews pass, Issues 002 and 003 are closed and
Issue 005 (Pipeline A training) and Issue 008 (Pipeline B data
prep) can start in parallel.
